# GRAST — WP07.4 CoT SFT (colab)

SFT từ checkpoint TinyStories 2.000 updates đã xác minh SHA256. Chỉ nạp model weights; optimizer/scaler/scheduler/RNG/data cursor khởi tạo mới cho stage SFT. **Không phải resume pretraining.** Model/tokenizer giữ nguyên. Mặc định arithmetic1500 train/300 validation, không dùng test hoặc GSM8K benchmark. Prompt và padding không tính loss; target gồm reasoning/final/EOS.

Mặc định `SMOKE_ONLY=True` dừng4 updates. Recipe gốc500 updates vẫn giữ nguyên. Train dài: `SMOKE_ONLY=False`; để tiếp tục smoke/mất runtime, điền `RESUME` tới checkpoint **SFT** latest, không điền checkpoint pretraining vào RESUME. Luôn stream log từng dòng.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

os.environ.update(PYTHONHASHSEED="42", CUBLAS_WORKSPACE_CONFIG=":4096:8")
REPO = Path("/content/grast-wp07-sft")
REVISION = "db0da482c3516d99b3fa6c3b6e6f566a65bf2c2f"  # verified WP07.4 implementation
SMOKE_ONLY = True
RESUME = ""  # full SFT latest.pt; empty means start SFT from pretrained weights


def run_logged(command, *, cwd=None):
    print("RUN:", " ".join(map(str, command)), flush=True)
    with subprocess.Popen(
        list(map(str, command)),
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        bufsize=1,
        env=os.environ.copy(),
    ) as process:
        try:
            for line in process.stdout:
                print(line, end="", flush=True)
            status = process.wait()
        except BaseException:
            import signal

            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                process.terminate()
            raise
        if status:
            raise subprocess.CalledProcessError(status, command)


def activate_project_source(repo):
    """Make a new editable install visible in this already-running kernel and its children."""
    import importlib
    import os
    import sys
    from pathlib import Path

    source = (Path(repo) / "src").resolve()
    package = source / "llm_scratch"
    required = ("__init__.py", "data/datasets.py", "data/sft_bundle.py", "training/sft.py")
    if not all((package / name).is_file() for name in required):
        raise FileNotFoundError("Missing SFT checkout; run the setup cell first: " + str(source))
    for name, module in list(sys.modules.items()):
        if name == "llm_scratch" or name.startswith("llm_scratch."):
            origin = getattr(module, "__file__", None)
            if origin is None or not Path(origin).resolve().is_relative_to(package):
                raise RuntimeError(
                    "Kernel already loaded llm_scratch from another location. "
                    "Restart the kernel and run setup again."
                )
    source_text = str(source)
    sys.path[:] = [source_text] + [entry for entry in sys.path if entry != source_text]
    inherited = [
        entry
        for entry in os.environ.get("PYTHONPATH", "").split(os.pathsep)
        if entry and entry != source_text
    ]
    os.environ["PYTHONPATH"] = os.pathsep.join([source_text, *inherited])
    importlib.invalidate_caches()
    for name in (
        "llm_scratch",
        "llm_scratch.data.datasets",
        "llm_scratch.data.sft_bundle",
        "llm_scratch.training.sft",
    ):
        module = importlib.import_module(name)
        if not Path(module.__file__).resolve().is_relative_to(package):
            raise RuntimeError("Imported unexpected project module: " + str(module.__file__))
    print("Kernel imports verified:", package, flush=True)
    return source


if not REPO.exists():
    run_logged(["git", "clone", "https://github.com/CuongBien/GRAST.git", REPO])
run_logged(["git", "fetch", "origin"], cwd=REPO)
run_logged(["git", "checkout", "--detach", REVISION], cwd=REPO)
run_logged([sys.executable, "-u", "-m", "pip", "install", "--no-deps", "-e", "."], cwd=REPO)
run_logged([sys.executable, "-u", "-m", "pip", "install", "matplotlib>=3.9,<4"], cwd=REPO)
run_logged(["git", "rev-parse", "HEAD"], cwd=REPO)
# pip writes an editable .pth file, but this kernel does not reload it automatically.
PROJECT_SOURCE = activate_project_source(REPO)
run_logged(
    [
        sys.executable,
        "-u",
        "-c",
        "from llm_scratch.data.datasets import file_sha256; "
        "from llm_scratch.data.sft_bundle import extract_bundle; "
        "from llm_scratch.training.sft import main; "
        "import llm_scratch; "
        "print('Subprocess imports verified:', llm_scratch.__file__, flush=True)",
    ],
    cwd=REPO,
)

In [ ]:
import torch

print("Python:", sys.version, "Torch:", torch.__version__, "CUDA:", torch.version.cuda, flush=True)
if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime; no silent CPU fallback")


def select_precision(capability_major, bf16_supported):
    return "bfloat16" if capability_major >= 8 and bf16_supported else "float16"


CAPABILITY = torch.cuda.get_device_capability(0)
PRECISION = select_precision(CAPABILITY[0], torch.cuda.is_bf16_supported())
print(
    "Training uses cuda:0:",
    torch.cuda.get_device_name(0),
    "precision:",
    PRECISION,
    "visible GPUs:",
    torch.cuda.device_count(),
    flush=True,
)

## Input và Drive

Chọn GPU T4/A100. Đưa `grast-wp07-sft-math-78f779a7d367.zip` và checkpoint pretraining `final.pt` vào `MyDrive/GRAST/wp07/sft/inputs/`. Có thể sửa `PRETRAIN_CHECKPOINT` để trỏ file sẵn trên Drive. Checkpoint local của project nằm tại `artifacts/checkpoints/wp07-pretrain-tinystories-2000/final.pt`. Khi resume SFT, không cần checkpoint pretraining cũ nữa.


In [ ]:
import tempfile

from google.colab import drive

from llm_scratch.data.datasets import file_sha256
from llm_scratch.data.sft_bundle import extract_bundle

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/GRAST/wp07/sft")
OUTPUT_ROOT = DRIVE_ROOT / "runs"
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)


BUNDLE_NAME = "grast-wp07-sft-math-78f779a7d367.zip"
BUNDLE_SHA256 = "100296172c05aff2bb99a4cc314cadf6dece5c65e0d7d7bb846a323794e7296e"
PRETRAIN_SHA256 = "27cf2d9cb9774b0023d4db09e2735fd666452f5a945468bc71722bb3c69b5309"

BUNDLE = DRIVE_ROOT / "inputs" / BUNDLE_NAME
PRETRAIN_CHECKPOINT = DRIVE_ROOT / "inputs/final.pt"
if not BUNDLE.is_file():
    raise FileNotFoundError("Upload SFT bundle to " + str(BUNDLE))
INPUTS = extract_bundle(
    BUNDLE, Path(tempfile.mkdtemp(prefix="grast-sft-")) / "verified", expected_sha256=BUNDLE_SHA256
)
if not RESUME:
    if not PRETRAIN_CHECKPOINT.is_file():
        raise FileNotFoundError("Upload pretrained final.pt to " + str(PRETRAIN_CHECKPOINT))
    if file_sha256(PRETRAIN_CHECKPOINT) != PRETRAIN_SHA256:
        raise ValueError("Pretraining checkpoint SHA256 mismatch")
print("Verified CoT inputs:", INPUTS, flush=True)

## Chạy

Recipe khởi đầu: model46,15M, max_length512 (không truncate), batch2/accumulation4, LR5e-5/warmup25,500 successful updates. Full validation và latest checkpoint mỗi25 updates; giữ mốc50/100/250/final500. Prompt và padding labels=-100; các examples tách theo batch, không concatenate. BF16 khi GPU hỗ trợ, ngược lại FP16; fresh warm-start được đổi runtime/precision, exact SFT resume phải giữ recipe/implementation/runtime tương thích. Nếu lỗi giữa update, resume từ latest hoàn tất trước đó.


In [ ]:
command = [
    sys.executable,
    "-u",
    "-m",
    "llm_scratch.training.sft",
    "--config",
    REPO / "configs/training/wp07_sft.toml",
    "--prepared-root",
    INPUTS / "sft_data",
    "--tokenizer",
    INPUTS / "tokenizer",
    "--device",
    "cuda",
    "--precision",
    PRECISION,
    "--output-root",
    OUTPUT_ROOT,
]
if SMOKE_ONLY:
    command += ["--stop-after", "4"]
if RESUME:
    if not Path(RESUME).is_file():
        raise FileNotFoundError(RESUME)
    command += ["--resume", RESUME]
else:
    command += ["--init-checkpoint", PRETRAIN_CHECKPOINT]
run_logged(command, cwd=REPO)

In [ ]:
import json
import zipfile

from IPython.display import Image, display

RUN = sorted(OUTPUT_ROOT.iterdir(), key=lambda p: p.name)[-1]
print("Run:", RUN, "Resume:", RUN / "checkpoints/latest.pt", flush=True)
report_path = RUN / "artifacts/sft_report.json"
if report_path.exists():
    report = json.loads(report_path.read_text())
    print(
        json.dumps(
            {k: report[k] for k in ("status", "step", "planned_steps", "tokens_optimized")},
            indent=2,
        ),
        flush=True,
    )
if (RUN / "artifacts/loss_curves.png").exists():
    display(Image(filename=str(RUN / "artifacts/loss_curves.png")))
SUMMARY = OUTPUT_ROOT.parent / (RUN.name + "-summary.zip")
with zipfile.ZipFile(SUMMARY, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN.rglob("*")):
        if path.is_file() and path.suffix not in {".pt", ".u32"}:
            archive.write(path, path.relative_to(RUN))
print("Summary:", SUMMARY, flush=True)
print("Keep latest/final/milestone weights for WP07.5; summary ZIP excludes weights.", flush=True)